# 02 — Baseline runs (Prompts 09–10): CoT and self-consistency only

Run **only after** notebook 01 passed and its outputs were checked (Prompt 08 STOP).
Kaggle: GPU T4 x2, Internet on. Produces the real baseline outputs the validation report (Prompt 10) needs.
No board runs here — those wait until the baseline is validated.

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("awesome-orchestrating-the-virtual-tumor-board"):
    subprocess.run(["git","clone","--depth","1","https://github.com/vibhum-phd/awesome-orchestrating-the-virtual-tumor-board.git"], check=True)
os.chdir("/kaggle/working/awesome-orchestrating-the-virtual-tumor-board/board-eval")
!pip install -q -r requirements.txt -r requirements-gpu.txt
print(subprocess.run(["git","rev-parse","--short","HEAD"],capture_output=True,text=True).stdout)

In [ ]:
# data (idempotent)
import os, subprocess
if not os.path.exists("data/raw/medqa_test.jsonl"):
    subprocess.run(["git","clone","--depth","1","https://github.com/gersteinlab/MedAgents.git","/tmp/MedAgents"], check=True)
    os.makedirs("data/raw", exist_ok=True)
    subprocess.run(["cp","/tmp/MedAgents/datasets/MedQA/test.jsonl","data/raw/medqa_test.jsonl"], check=True)
!python scripts/prepare_data.py --src data/raw/medqa_test.jsonl --out data/processed > /dev/null
if not os.path.exists("data/processed/medmcqa_dev_onc.jsonl"):
    !python scripts/prepare_extra_data.py --out data/processed
!wc -l data/processed/*.jsonl

## 1. External anchor: CoT on MedQA 4-option test (compare with published Qwen2.5-7B numbers)

In [ ]:
!python scripts/run_experiment.py --dataset medqa4_all --method cot --seed 0 --backend vllm --chunk 128

## 2. Baselines on the oncology sets, seed 0

In [ ]:
for ds in ["medqa_onc", "medmcqa_onc"]:
    for m in ["cot", "sc"]:
        !python scripts/run_experiment.py --dataset {ds} --method {m} --seed 0 --backend vllm --chunk 64

## 3. Determinism check: rerun CoT on the first 20 MedQA-onc items into a separate folder

In [ ]:
!python scripts/run_experiment.py --dataset medqa_onc --method cot --seed 0 --backend vllm --limit 20 --out results/rerun

## 4. Baseline report (numbers only)

In [ ]:
!python scripts/baseline_report.py --raw results/raw --rerun results/rerun --out results/baseline
!cat results/raw/run_meta.jsonl | tail -3

In [ ]:
# Every unparseable answer, for manual inspection (Prompt 10/11)
import json, glob
for fp in sorted(glob.glob("results/raw/*__cot__*.jsonl")):
    bad = [json.loads(l) for l in open(fp) if json.loads(l)["final"] is None]
    print(fp, "parse failures:", len(bad))
    for r in bad[:5]:
        print("   ...", r["samples"][0]["text"][-300:].replace("\n", " | "))

In [ ]:
!cd results && zip -qr /kaggle/working/baseline_outputs.zip raw rerun baseline && ls -la /kaggle/working/baseline_outputs.zip

## STOP (Prompt 10)
Share `baseline_outputs.zip`. The baseline validation report is completed from these real numbers before
any board run (notebook 03).